In [ ]:
%pip -q install -U bitsandbytes transformers accelerate datasets peft trl matplotlib

In [ ]:
import os, re, math
import numpy as np
import matplotlib.pyplot as plt
from tqdm import tqdm

from dotenv import load_dotenv
from huggingface_hub import login

import torch
import torch.nn.functional as F

from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig, set_seed
from datasets import load_dataset
from peft import PeftModel

# Load env vars from .env (Cursor/local-friendly)
load_dotenv(override=True)

# Hugging Face auth (no google.colab)
hf_token = os.getenv("HF_TOKEN")
assert hf_token, "Missing HF_TOKEN. Add it to your .env file or environment variables."
login(hf_token, add_to_git_credential=True)

%matplotlib inline

In [ ]:
BASE_MODEL = "meta-llama/Llama-3.2-3B"

LITE_MODE = True
DATASET_NAME = "ed-donner/items_prompts_lite" if LITE_MODE else "ed-donner/items_prompts_full"

# Your LoRA adapter on HF (this is the “evidence” of fine-tuning)
# Example from your earlier runs:
FINETUNED_MODEL = "ed-donner/price-2025-11-30_15.10.55-lite"
REVISION = None  # optional: pin to a commit hash

QUANT_4_BIT = True  # 4-bit = fits T4; set False for 8-bit if you have VRAM

MAX_NEW_TOKENS = 8
TEST_SIZE = 250   # keep it manageable

In [ ]:
login(hf_token, add_to_git_credential=True)

ds = load_dataset(DATASET_NAME)
test = ds["test"]

print("✓ Loaded test:", len(test))
print(test[0])

In [ ]:
import os, re, math
import numpy as np
import matplotlib.pyplot as plt
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, set_seed
from datasets import load_dataset
from peft import PeftModel

device = torch.device("cpu")
print("Device:", device)

# ---- CONFIG ----
BASE_MODEL = "meta-llama/Llama-3.2-3B"
DATASET_NAME = "ed-donner/items_prompts_lite"  # use lite for CPU speed
FINETUNED_MODEL = "ed-donner/price-2025-11-30_15.10.55-lite"
REVISION = None  # optional commit hash
MAX_NEW_TOKENS = 8
TEST_SIZE = 25   # keep small on CPU

# Load dataset
ds = load_dataset(DATASET_NAME)
test = ds["test"]

# Load tokenizer + base model on CPU (no quantization)
tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL, trust_remote_code=True)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

base_model = AutoModelForCausalLM.from_pretrained(BASE_MODEL)
base_model.to(device)
base_model.eval()

# Attach LoRA adapter (still works on CPU)
if REVISION:
    fine_tuned_model = PeftModel.from_pretrained(base_model, FINETUNED_MODEL, revision=REVISION)
else:
    fine_tuned_model = PeftModel.from_pretrained(base_model, FINETUNED_MODEL)

fine_tuned_model.to(device)
fine_tuned_model.eval()

print("✅ Loaded LoRA adapter:", FINETUNED_MODEL)

In [ ]:
# --- Fast CPU evaluation (tiny test size + progress) ---
from tqdm import tqdm

TEST_SIZE = 10         # ✅ keep tiny on CPU (try 10 if you’re patient)
MAX_NEW_TOKENS = 6     # ✅ shorter generation = faster

NUM_RE = re.compile(r"\d+(\.\d+)?")

def extract_number(text: str) -> float:
    text = text.replace(",", "")
    m = NUM_RE.search(text)
    return float(m.group(0)) if m else float("nan")

def predict_price(model, prompt: str) -> float:
    inputs = tokenizer(prompt, return_tensors="pt")
    with torch.no_grad():
        out_ids = model.generate(
            **inputs,
            max_new_tokens=MAX_NEW_TOKENS,
            do_sample=False,   # greedy (deterministic)
        )
    prompt_len = inputs["input_ids"].shape[1]
    gen = tokenizer.decode(out_ids[0, prompt_len:], skip_special_tokens=True).strip()
    return extract_number(gen)

# Evaluate on small subset
set_seed(42)
subset = [test[i] for i in range(min(TEST_SIZE, len(test)))]

y_true = np.array([float(x["completion"]) for x in subset])

y_pred = []
for dp in tqdm(subset, desc="Predicting"):
    y_pred.append(predict_price(fine_tuned_model, dp["prompt"]))
y_pred = np.array(y_pred, dtype=float)

mask = ~np.isnan(y_pred)
mae = float(np.mean(np.abs(y_true[mask] - y_pred[mask])))

print(f"CPU Eval on {mask.sum()}/{len(subset)} parsed predictions")
print(f"MAE: ${mae:,.2f}")

# Plot Pred vs Actual (only if we have >=2 valid points)
if mask.sum() >= 2:
    plt.figure()
    plt.scatter(y_true[mask], y_pred[mask], s=40, alpha=0.8)
    maxv = float(max(y_true[mask].max(), y_pred[mask].max()))
    plt.plot([0, maxv], [0, maxv], lw=2, alpha=0.5)
    plt.xlabel("Actual Price ($)")
    plt.ylabel("Predicted Price ($)")
    plt.title("Fine-tuned (LoRA) — Pred vs Actual (CPU tiny subset)")
    plt.show()
else:
    print("Not enough valid numeric predictions to plot.")

In [ ]:
from tqdm import tqdm
import numpy as np
import matplotlib.pyplot as plt

def eval_on_subset(model, subset, label):
    preds = []
    for dp in tqdm(subset, desc=f"Predicting ({label})"):
        preds.append(predict_price(model, dp["prompt"]))
    preds = np.array(preds, dtype=float)
    truth = np.array([float(dp["completion"]) for dp in subset], dtype=float)

    mask = ~np.isnan(preds)
    mae = float(np.mean(np.abs(truth[mask] - preds[mask]))) if mask.any() else float("nan")
    return truth, preds, mask, mae

# Use the same subset you already built
y_true_b, y_base, mask_b, base_mae = eval_on_subset(base_model, subset, "Base")
y_true_f, y_ft,   mask_f, ft_mae   = eval_on_subset(fine_tuned_model, subset, "Fine-tuned")

print(f"Base MAE:       ${base_mae:,.2f}")
print(f"Fine-tuned MAE: ${ft_mae:,.2f}")
print("✅ Fine-tuning helped!" if ft_mae < base_mae else "⚠️ No improvement on this tiny subset (try TEST_SIZE=10)")